# 03 — Dataset Split

Creates a reproducible stratified 80/10/10 train/validation/test split from `results/balanced_subset.csv`.

**Random seed:** 42  
**Stratification:** by cultivar  
**Output:** `results/train.csv`, `results/validation.csv`, `results/test.csv`, `results/dataset_split.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
RESULTS_DIR = Path("../results")
DATASET_DIR = Path("../../Dataset")

## 1. Load Balanced Subset

In [ ]:
df = pd.read_csv(RESULTS_DIR / "balanced_subset.csv")
print(f"Rows loaded : {len(df)}")
print(f"Columns     : {df.columns.tolist()}")
df.head()

## 2. Validate Input

In [ ]:
assert "image" in df.columns, "Column 'image' missing"
assert "cultivar" in df.columns, "Column 'cultivar' missing"

dupes = df["image"].duplicated().sum()
assert dupes == 0, f"Duplicate image entries: {dupes}"

missing_on_disk = [img for img in df["image"] if not (DATASET_DIR / img).exists()]
assert len(missing_on_disk) == 0, f"{len(missing_on_disk)} images missing from disk"

multi_label = df.groupby("image")["cultivar"].nunique()
assert (multi_label > 1).sum() == 0, "Some images have multiple cultivar labels"

print("All input validation checks passed.")
print(f"  Images    : {len(df)}")
print(f"  Cultivars : {df['cultivar'].nunique()}")

## 3. Analyze Class Sizes

In [ ]:
counts = df["cultivar"].value_counts()
print(f"Min images per cultivar  : {counts.min()}")
print(f"Max images per cultivar  : {counts.max()}")
print(f"Mean images per cultivar : {counts.mean():.1f}")

small = counts[counts < 10]
if len(small) > 0:
    print(f"\nWARNING: {len(small)} cultivar(s) have < 10 images — split counts may deviate from 80/10/10")
    print(small)
else:
    print("\nAll cultivars have >= 10 images. Stratified 80/10/10 split is safe.")

## 4. Perform Stratified Split

In [ ]:
# 80% train, 20% temp
train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df["cultivar"], random_state=RANDOM_SEED
)
# 50% of temp → val, 50% → test  (each = 10% of total)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["cultivar"], random_state=RANDOM_SEED
)

print(f"Train      : {len(train_df)} images")
print(f"Validation : {len(val_df)} images")
print(f"Test       : {len(test_df)} images")
print(f"Total      : {len(train_df) + len(val_df) + len(test_df)} images")

## 5. Save Split Manifests

In [ ]:
for name, split in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    out = split[["image", "cultivar"]].copy()
    out["split"] = name
    out.to_csv(RESULTS_DIR / f"{name}.csv", index=False)
    print(f"Saved {name}.csv ({len(out)} rows)")

master = pd.concat([
    train_df[["image", "cultivar"]].assign(split="train"),
    val_df[["image", "cultivar"]].assign(split="validation"),
    test_df[["image", "cultivar"]].assign(split="test"),
], ignore_index=True)
master.to_csv(RESULTS_DIR / "dataset_split.csv", index=False)
print(f"Saved dataset_split.csv ({len(master)} rows)")

## 6. Verify No Leakage

In [ ]:
train_imgs = set(train_df["image"])
val_imgs   = set(val_df["image"])
test_imgs  = set(test_df["image"])

assert len(train_imgs & val_imgs)  == 0, "LEAKAGE: train ∩ validation"
assert len(train_imgs & test_imgs) == 0, "LEAKAGE: train ∩ test"
assert len(val_imgs   & test_imgs) == 0, "LEAKAGE: validation ∩ test"

assert (train_imgs | val_imgs | test_imgs) == set(df["image"]), \
    "Split images do not match balanced subset"

for name, split in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    assert split["image"].duplicated().sum() == 0, f"Duplicates within {name}"

print("train ∩ validation = empty  ✓")
print("train ∩ test       = empty  ✓")
print("validation ∩ test  = empty  ✓")
print("train + val + test = balanced subset  ✓")
print("No duplicates within any split  ✓")

## 7. Analyze Split Distributions

In [ ]:
train_counts = train_df["cultivar"].value_counts().rename("Train")
val_counts   = val_df["cultivar"].value_counts().rename("Validation")
test_counts  = test_df["cultivar"].value_counts().rename("Test")

dist = pd.concat([train_counts, val_counts, test_counts], axis=1).fillna(0).astype(int)
dist["Total"] = dist.sum(axis=1)
dist.index.name = "Cultivar"
dist = dist.reset_index()
dist.to_csv(RESULTS_DIR / "split_class_distribution.csv", index=False)

print(f"Train   — min: {dist['Train'].min()}, max: {dist['Train'].max()}, mean: {dist['Train'].mean():.1f}")
print(f"Val     — min: {dist['Validation'].min()}, max: {dist['Validation'].max()}, mean: {dist['Validation'].mean():.1f}")
print(f"Test    — min: {dist['Test'].min()}, max: {dist['Test'].max()}, mean: {dist['Test'].mean():.1f}")
dist.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

splits  = ["Training", "Validation", "Test"]
n_imgs  = [len(train_df), len(val_df), len(test_df)]
colors  = ["#4878CF", "#6ACC65", "#D65F5F"]

axes[0].bar(splits, n_imgs, color=colors, width=0.5)
axes[0].set_title("Images per Split")
axes[0].set_ylabel("Number of Images")
for i, v in enumerate(n_imgs):
    axes[0].text(i, v + 5, str(v), ha="center", va="bottom", fontsize=10)
axes[0].set_ylim(0, max(n_imgs) * 1.12)

tc = train_df["cultivar"].value_counts().sort_index()
vc = val_df["cultivar"].value_counts().sort_index()
xc = test_df["cultivar"].value_counts().sort_index()

axes[1].plot(range(len(tc)), tc.values, color="#4878CF", linewidth=1, label=f"Train (n={len(train_df)})")
axes[1].plot(range(len(vc)), vc.values, color="#6ACC65", linewidth=1, label=f"Validation (n={len(val_df)})")
axes[1].plot(range(len(xc)), xc.values, color="#D65F5F", linewidth=1, label=f"Test (n={len(test_df)})")
axes[1].set_title("Per-Cultivar Image Counts Across Splits")
axes[1].set_xlabel("Cultivar (sorted)")
axes[1].set_ylabel("Number of Images")
axes[1].legend()

fig.tight_layout()
fig.savefig(RESULTS_DIR / "split_distribution.png", dpi=150)
plt.show()
print("Saved split_distribution.png")

## 8. Display Summary

In [ ]:
total = len(train_df) + len(val_df) + len(test_df)
summary_rows = [
    ("Total images",        total),
    ("Training images",     len(train_df)),
    ("Validation images",   len(val_df)),
    ("Test images",         len(test_df)),
    ("Number of cultivars", train_df["cultivar"].nunique()),
    ("Training percentage", f"{len(train_df)/total*100:.2f}%"),
    ("Validation percentage", f"{len(val_df)/total*100:.2f}%"),
    ("Test percentage",     f"{len(test_df)/total*100:.2f}%"),
    ("Random seed",         RANDOM_SEED),
]
summary_df = pd.DataFrame(summary_rows, columns=["Metric", "Value"])
summary_df.to_csv(RESULTS_DIR / "split_summary.csv", index=False)
summary_df

## 9. Final Validation Checks

In [ ]:
# Reload from disk and re-verify
train_reload = pd.read_csv(RESULTS_DIR / "train.csv")
val_reload   = pd.read_csv(RESULTS_DIR / "validation.csv")
test_reload  = pd.read_csv(RESULTS_DIR / "test.csv")
master_reload = pd.read_csv(RESULTS_DIR / "dataset_split.csv")

assert len(train_reload) == 4000
assert len(val_reload)   == 500
assert len(test_reload)  == 500
assert len(master_reload) == 5000
assert set(master_reload["split"].unique()) == {"train", "validation", "test"}

# Every cultivar present in all three splits
assert train_reload["cultivar"].nunique() == 100
assert val_reload["cultivar"].nunique()   == 100
assert test_reload["cultivar"].nunique()  == 100

# No cross-split overlap
t = set(train_reload["image"])
v = set(val_reload["image"])
x = set(test_reload["image"])
assert len(t & v) == 0
assert len(t & x) == 0
assert len(v & x) == 0

print("Final validation checks passed.")
print(f"  train.csv      : {len(train_reload)} rows, {train_reload['cultivar'].nunique()} cultivars")
print(f"  validation.csv : {len(val_reload)} rows, {val_reload['cultivar'].nunique()} cultivars")
print(f"  test.csv       : {len(test_reload)} rows, {test_reload['cultivar'].nunique()} cultivars")
print(f"  dataset_split  : {len(master_reload)} rows, splits: {sorted(master_reload['split'].unique())}")
print("\nDataset split is complete and verified. Ready for image preprocessing.")